# Interactive charts & dashboards

_Data Wrangling_

**Let the reader hover, zoom, and filter — and know when a still picture says it better.**

Every chart is a function from data to pixels. A static chart applies that function
       once, with choices you fixed: these axes, this zoom, this filter. An interactive chart keeps
       the function live, so the reader can change an input &mdash; the zoom window, a filter value, a hovered
       point &mdash; and the picture redraws.

       That is the whole idea, and it cuts both ways. Live re-rendering is wonderful for exploration
       and for letting many people ask their own questions. It is a liability when you need one fixed,
       reproducible, printable artifact. So the skill here is not "make it interactive" &mdash; it is
       choosing interactive versus static for the job in front of you, and, when you go interactive,
       choosing the right tool and not drowning the message in controls.

---

This notebook builds the idea up one step at a time: a hover-and-zoom Plotly chart, then a tiny filterable Streamlit dashboard, then a real multi-class scatter on wine data. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, plus a class label.

In [ ]:
from sklearn.datasets import load_wine

data = load_wine(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target classes :", list(data.target_names))
data.frame.head()

## Reference implementation — pandas + plotly + streamlit

We build the same idea in two pieces: first an **interactive single chart** with Plotly Express (hover, zoom, pan in the browser), then a small **Streamlit dashboard** that wires filter widgets to a chart and a table so the reader can slice the data themselves.

### Step 1 — A small products table to chart

Before any chart we need data. We build a tiny real-ish products table with a name, a category, a price, and a rating for eight products. Keeping it small means every dot on the scatter is individually readable when we hover it.

In [ ]:
# pip install plotly streamlit pandas
# A small real-ish products table (price, rating, category, name).
import pandas as pd
import plotly.express as px

df = pd.DataFrame({
    "name":     ["Echo", "Bolt", "Nimbus", "Pixel", "Quartz", "Drift", "Sol", "Vega"],
    "category": ["Audio", "Tools", "Audio", "Phones", "Tools", "Audio", "Phones", "Phones"],
    "price":    [49, 120, 89, 699, 35, 59, 999, 450],
    "rating":   [4.6, 3.1, 4.2, 4.8, 2.7, 3.9, 2.9, 4.4],
})

### Step 2 — Make one interactive chart with Plotly

Plotly Express turns the table into a scatter where the reader can zoom, pan, and hover. Two arguments do the real work: `color=` encodes category as a third dimension on top of the x/y price and rating, and `hover_data=` decides which row fields appear in the tooltip when you point at a dot.

In [ ]:
# Interactive single chart (Plotly Express).
# color= adds a 3rd dimension; hover_data= lets the reader read the exact row.
fig = px.scatter(
    df, x="price", y="rating",
    color="category",                 # third dimension via color
    hover_data=["name", "category"],  # tooltip shows the row on hover
    title="Price vs rating (hover a dot to read its name)",
)
fig.show()   # zoom / pan / hover all work in the browser

### Step 3 — Wire a tiny Streamlit dashboard

A single chart is fixed; a dashboard lets the reader change inputs. This skeleton adds a category dropdown and a minimum-rating slider, filters the table by whatever the reader picks, and then redraws **both** a chart and a table from that filtered view. Save it as `app.py` and run `streamlit run app.py` to see the chart and table update together as you move the controls.

In [ ]:
# Tiny dashboard skeleton (Streamlit).
# Save as app.py and run:  streamlit run app.py
import streamlit as st

st.title("Product explorer")

# --- filter widgets ---
cat = st.selectbox("Category", ["All"] + sorted(df["category"].unique()))
min_rating = st.slider("Minimum rating", 0.0, 5.0, 0.0, 0.1)

# --- apply filters ---
view = df[df["rating"] >= min_rating]
if cat != "All":
    view = view[view["category"] == cat]

# --- linked chart + table update together as the filters change ---
chart = px.scatter(view, x="price", y="rating",
                   color="category", hover_data=["name"])
st.plotly_chart(chart, use_container_width=True)
st.dataframe(view)

## Visualize the data & results

_On real wine-chemistry data, plot alcohol vs color intensity colored by cultivar — the kind of multi-class scatter you'd ship as an interactive Plotly chart (hover the row, filter by cultivar). Here it's a static stand-in._

### Step 1 — Load and subsample the wine data

Real datasets are too dense to plot every point legibly, so we load the 178-wine dataset and take a random sample of 54 rows. Sorting by `alcohol` keeps the printed output tidy. We then pull the three series we need: alcohol (x), color intensity (y), and the cultivar label (which becomes color).

In [ ]:
import numpy as np
from sklearn.datasets import load_wine

d = load_wine(as_frame=True)
df = d.frame

# Subsample 54 wines for a readable scatter (<= 60 plotted points).
sub = df.sample(n=54, random_state=0).sort_values('alcohol')
x   = sub['alcohol'].to_numpy()           # x-axis
y   = sub['color_intensity'].to_numpy()   # y-axis
cls = sub['target'].to_numpy()            # cultivar 0/1/2 -> color

### Step 2 — Group the points by cultivar

What `color=` does for you in one Plotly call, we do by hand here: split the points into the three cultivars so each class is its own series. The commented Plotly snippet at the bottom shows the single interactive call this stands in for — where hovering reveals the full row and clicking a legend entry filters a cultivar in or out.

In [ ]:
# Group the points by cultivar (what color= does in px.scatter).
for c in [0, 1, 2]:
    pts = [(round(float(a), 3), round(float(b), 3))
           for a, b, k in zip(x, y, cls) if k == c]
    print(f"cultivar {c}: n={len(pts)}")
    print(pts)

# In Plotly this would be one interactive call:
#   import plotly.express as px
#   px.scatter(sub, x='alcohol', y='color_intensity', color='target',
#              hover_data=df.columns)   # hover -> full row; legend click -> filter

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** You found a striking single fact: in your A/B test, the new layout lifted conversion from 4.0% to 4.8%. You need to put it in the launch slide deck and the exec summary. Static chart or interactive dashboard?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Identify the message: it is a single, clear comparison (one number versus another). — _Interactivity helps when readers must explore many slices; here the point is one fact, so there is nothing to drill into._
- Note the medium: slides and a written summary. — _Clicks, hover, and zoom do not work in a printed deck or a static document._
- Pick a clean static bar chart (two bars, the lift labeled) and export it as an image. — _Reproducible, printable, and the headline lands at a glance._

**Answer:** Use a static two-bar chart with the lift labeled. The message is a single clear fact, the medium is print/slides, and you want the exact same artifact every time. An interactive dashboard would add clicks and controls that only distract from the one number that matters.

</details>

**Problem 2.** An analytics team keeps pinging you for "the same chart but filtered to my region / my product / last 7 days." You re-run the notebook and re-send a PNG each time. What should you build instead, and what are the two main risks?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recognize the pattern: many people want slightly different slices of one dataset, on demand. — _That is exactly the self-serve case where a dashboard beats one-off static charts._
- Build a small dashboard (Streamlit / Dash / Panel, or a BI tool) with region, product, and date filters feeding a chart and a table. — _Each analyst filters to their own case without asking you, freeing you from the re-render loop._
- Plan for maintenance and scale: give it an owner plus a freshness check, and aggregate/subsample before plotting. — _An unmaintained dashboard shows stale or wrong numbers; a heavy plot over the full data freezes the browser._

**Answer:** Build a self-serve dashboard with region/product/date filters (Streamlit, Plotly Dash, Panel, or a BI tool like Tableau/Power BI/Looker). The two main risks are nobody maintaining it (so it silently goes stale &mdash; give it an owner and a freshness indicator) and heavy/slow plots on big data (aggregate or subsample before rendering).

</details>

**Problem 3.** You build a beautiful Plotly scatter where the key insight — one outlier cluster — only becomes visible after the reader zooms into the bottom-left and hovers. A color-blind teammate also says they can't tell two of your series apart. Name the two pitfalls and the fixes.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Notice the insight is hidden behind interaction: the default view doesn't show it. — _Most readers never zoom; if the headline needs a click to appear, most readers miss it._
- Make the default view already show the cluster (pre-zoom to it, annotate it, or add a static inset). — _Interactivity should be for going deeper, not for finding the main point._
- Fix the color-only encoding: use a color-blind-safe palette and add redundant shape/text labels. — _Color alone fails for color-blind readers; redundant encodings keep the chart accessible._

**Answer:** Two pitfalls: hiding the key message behind clicks (fix: make the default view already tell the story &mdash; pre-zoom or annotate the cluster) and accessibility (fix: a color-blind-safe palette plus redundant shape/text labels, not color alone). Interactivity is for drilling deeper, never for revealing the headline.

</details>